## Assignment 1: Prompting for Program Repair

In this assignment, you will design prompts to repair buggy Python functions and evaluate the quality of the bug repair using unit tests.

What you will do:
- Use this notebook as a guide to complete your assignment.
- Inside the `data` directory, you’ll find one folder per buggy Python function (`data/BUGxxx/`), each with example tests in `test_bug.py`.
- Experiment with prompting techniques using the model (Gemma 3 270M IT), and observe how variations in prompt wording affect bug repair performance.
- For each buggy Python function, record the prompts you used, then choose one final repaired code to submit.

Submission Instructions:  
Submit a ZIP file of the `results` folder, containing the two items listed below, named `SWE270P_A1.ZIP`.  
**Please make sure to strictly follow the submission format and filename, as an autograder will be used for grading. <br> An incorrect format or filename may result in a score of 0.**  
 1) `prompts.jsonl`: all the prompts you used. This notebook includes a helper function that logs them to `results/prompts.jsonl`.
 2) `BUGxxx.py` files: one final repaired code file per buggy Python function (e.g., `BUG001.py`), saved under `results/final_patches/`.  


Required File Structure:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

Helpful Guidelines:
- Use `test_bug.py` to evaluate the quality of repaired code.
- Add your own test cases to check the robustness of your repaired code.
- Final submissions will be graded using an autograder. Do not modify any function names or parameter definitions.
- Each final patch must be a complete replacement for that bug's `buggy.py`. Keep its imports and helper functions, not only the fixed function (e.g., BUG001 needs `get` and `insert_or_update`).

Recommended process:
1) Work on one bug at a time (e.g., BUG001).
2) Select a prompting technique (e.g., zero-shot, chain-of-thought, tests-aware, or self-refine).
3) Using the selected technique, draft your prompt and call the model.
4) Copy the repaired code into that bug's `buggy.py` and run its tests with `test_bug.py` (e.g., in `data/BUG001/`). <br>
    **For example**, for BUG001, run `pytest test_bug.py` in a terminal inside `data/BUG001/`, or run `!cd data/BUG001 && pytest test_bug.py` in a notebook cell.
5) Modify the prompt using different wording and structure, or try another prompting technique until the repaired code passes all tests.   
6) Log the prompting technique and the prompt used to call the model in `prompts.jsonl`.
7) Save your final repaired code for each bug (e.g., `BUG001.py`) for submission.



## 0. Setup (Required: Gemma 3 270M IT)

This assignment **must use** `google/gemma-3-270m-it` via Hugging Face Transformers (local CPU/GPU). No external APIs are needed.

We recommend using Google Colab. If you're using Colab, make sure you have a GPU runtime (Runtime → Change runtime type → select T4 GPU). If you run locally, use Python 3.10 or newer.

### Completed local workflow

Use the Python 3.12 virtual environment at `../.venv`. The cells below execute a resumable local experiment. Completed attempt IDs reuse their actual saved results; a fresh output directory performs new model inference. All raw completions, including failures, are preserved. Final target statements must match a logged Gemma response.

The assistant supplies detailed repair plans, including Python expressions and statement-level code plans. This is guided implementation, not a claim that the 270M model independently diagnoses the bugs. Original imports, helpers, signatures, and documentation are retained by a separate assembler.

```bash
source ../.venv/bin/activate
python execute_notebook.py
```

`execute_notebook.py` executes these Python-only cells sequentially in the venv and records outputs without starting a Jupyter kernel server. The same cells can be run interactively with a venv notebook kernel. See `README.md` for setup and individual commands.


In [1]:
import sys
from pathlib import Path

assert sys.prefix != sys.base_prefix, "Select the assignment venv."
assert Path("data").is_dir(), "Run from assignment/assignment1/SWE270P_A1."
print("Python:", sys.executable)
print("Virtual environment:", sys.prefix)

Python: /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/../.venv/bin/python
Virtual environment: /Users/ericsong/Courses/SWE270P/assignment/assignment1/.venv


In [2]:
from importlib.metadata import version

for package in ("torch", "transformers", "pytest", "huggingface-hub", "accelerate"):
    print(f"{package}=={version(package)}")

torch==2.14.1
transformers==5.20.2
pytest==9.1.1
huggingface-hub==2.2.0
accelerate==1.15.0


### Model access and local cache

The completed run uses cached weights for the required [`google/gemma-3-270m-it`](https://huggingface.co/google/gemma-3-270m-it) model. No token is stored in this project. On a fresh machine, accept the model terms, authenticate locally with `hf auth login`, and cache the model with `hf download google/gemma-3-270m-it`.

The implementation uses local CPU float32 inference through Transformers. Sampling uses temperature 1.0, top-p 0.95, top-k 50, and logged seeds. Invalid syntax or logical nesting is rejected and retried; no generated algorithm is manually repaired after generation.

In [3]:
from huggingface_hub import try_to_load_from_cache

cached = try_to_load_from_cache("google/gemma-3-270m-it", "config.json")
assert isinstance(cached, str), "Cache the required model before running."
print("Required model is cached locally.")

Required model is cached locally.


In [4]:
from workflow import MODEL_ID, generate_text, load_model

# generate_text applies the chat template, samples locally, and decodes only
# new tokens. Its complete implementation is in workflow.py.
tokenizer, model = load_model()
print("Model:", MODEL_ID, "Device:", model.device, "Dtype:", model.dtype)
# New calls go through run_prompt so raw prompts and completions are logged.
# The experiment sections below call the model only for uncompleted IDs.

Model: google/gemma-3-270m-it Device: cpu Dtype: torch.float32


W1019 20:42:52.447000 23844 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|##########| 236/236 [00:00<00:00, 344.93it/s]


## Prompting Techniques: Zero‑Shot, One‑Shot, Chain‑of‑Thought, and Few‑Shot  
**Goal:** Practice different prompting strategies and compare their effects on bug repair correctness.

After generating the fix for a problem, you can paste it into the corresponding `buggy.py` and run `pytest` for that folder to see if it passes.

In [5]:
import json
from workflow import (
    ROOT, PROMPTS_PATH, PATCH_DIR, extract_code, assemble_patch,
    log_prompt, run_prompt, save_final_patch,
)
from prompt_designs import TECHNIQUES
from structured_plans import make_code_plan_prompt
import run_experiments

def run_method(method):
    """Run/resume two real designs for each of the ten target bugs."""
    run_experiments.main(["--format", "code-plan", "--method", method])

def saved_attempt(bug, method, variant):
    attempt_id = f"{bug}_{method}_v{variant}_code-plan_primary"
    return next(
        json.loads(line) for line in PROMPTS_PATH.read_text().splitlines()
        if json.loads(line)["attempt_id"] == attempt_id
    )

def display_attempt(bug, method, variant):
    record = saved_attempt(bug, method, variant)
    print("ATTEMPT:", record["attempt_id"], "SEED:", record["seed"])
    print("PROMPT:\n", record["prompt"])
    print("RAW GEMMA COMPLETION:\n", record["completion"])
    return record

print("Inference, raw logging, extraction and scaffold-preserving assembly are implemented.")

Inference, raw logging, extraction and scaffold-preserving assembly are implemented.


## 1. Zero‑Shot
A prompting technique that asks the model to solve the task **without** examples. Include specific output constraints.

In [6]:
run_method("zero-shot")
example_zero_shot = display_attempt("BUG000", "zero-shot", 1)
example_zero_shot_prompt = example_zero_shot["prompt"]
example_zero_shot_output = example_zero_shot["completion"]

Selected bugs: BUG000, BUG001, BUG002, BUG003, BUG004, BUG005, BUG006, BUG007, BUG008, BUG009
ATTEMPT: BUG000_zero-shot_v1_code-plan_primary SEED: 4900
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Target function signature: def has_close_elements(numbers: List[float], threshold: float) -> bool:
The following Python statements specify the repair plan. Emit the function body, preserving these operations, names, and relative indentation:
```python
for i in range(len(numbers)):
    for j in range(i + 1, len(numbers)):
        if abs(numbers[i] - numbers[j]) < threshold:
            return True
return False
```
Return only the Python implementation for has_close_elements in a code block. No docstrings, example functions, or test calls.

RAW GEMMA COMPLETION:
 ```python
def has_close_elements(numbers: List[float], threshold: float) -> bool:
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers))

### Logging and complete-module assembly

Every new inference call is logged before extraction or tests. Failures are retained in `results/prompts.jsonl`. The one interrupted exploratory call is explicitly marked without an invented completion.

`extract_code` selects emitted Python. `assemble_patch` replaces the target function while preserving the original imports, helpers, signature, and documentation. For body-only output it supplies the unchanged original header and a uniform indentation shift. Executable statements and relative nesting must match the raw response. Misplaced returns and other invalid nesting are rejected, not corrected.

`save_final_patch("BUG000.py", raw_response)` implements the same assembly, but final selection in the runner occurs only after the full test suite for that bug passes. Placeholder filenames are rejected. The package builder proves every final file is exactly reconstructible from its selected response.

In [7]:
your_zero_shot = display_attempt("BUG000", "zero-shot", 2)
your_zero_shot_prompt = your_zero_shot["prompt"]
your_zero_shot_output = your_zero_shot["completion"]

ATTEMPT: BUG000_zero-shot_v2_code-plan_primary SEED: 4901
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Target function signature: def has_close_elements(numbers: List[float], threshold: float) -> bool:
Review this statement-level implementation plan for the target. Output its complete function, or just its body. Preserve the plan's control flow and final return outside all loops:
```python
for i in range(len(numbers)):
    for j in range(i + 1, len(numbers)):
        if abs(numbers[i] - numbers[j]) < threshold:
            return True
return False
```
Previous generation failed: Unexpected top-level executable code in the response.. Use the exact signature and valid Python indentation; do not emit a docstring.
Return only the Python implementation for has_close_elements in a code block. No docstrings, example functions, or test calls.

RAW GEMMA COMPLETION:
 ```python
def has_close_elements(numbers: List[float]

## 2. One‑Shot
A prompting technique that includes **one worked example** before the target task.

In [8]:
run_method("one-shot")
example_one_shot = display_attempt("BUG002", "one-shot", 1)
example_one_shot_prompt = example_one_shot["prompt"]
example_one_shot_output = example_one_shot["completion"]

Selected bugs: BUG000, BUG001, BUG002, BUG003, BUG004, BUG005, BUG006, BUG007, BUG008, BUG009
ATTEMPT: BUG002_one-shot_v1_code-plan_primary SEED: 5110
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Worked example 1:
Plan: set answer=0; for each x in items add x to answer; after the loop return answer.
Solution:
```python
def total(items):
    answer = 0
    for x in items:
        answer += x
    return answer
```
Target function signature: def shortest_paths(source, weight_by_edge):
The following Python statements specify the repair plan. Emit the function body, preserving these operations, names, and relative indentation:
```python
distances = {source: 0}
for u, v in weight_by_edge:
    distances.setdefault(u, float('inf'))
    distances.setdefault(v, float('inf'))
for _ in range(len(distances) - 1):
    for (u, v), weight in weight_by_edge.items():
        distances[v] = min(distances[v], distances[u] + weight

In [9]:
your_one_shot = display_attempt("BUG002", "one-shot", 2)
your_one_shot_prompt = your_one_shot["prompt"]
your_one_shot_output = your_one_shot["completion"]

ATTEMPT: BUG002_one-shot_v2_code-plan_primary SEED: 5111
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Worked example 1:
Plan: if x is None return 0; otherwise return x.
Solution:
```python
def value_or_zero(x):
    if x is None:
        return 0
    return x
```
Target function signature: def shortest_paths(source, weight_by_edge):
Review this statement-level implementation plan for the target. Output its complete function, or just its body. Preserve the plan's control flow and final return outside all loops:
```python
distances = {source: 0}
for u, v in weight_by_edge:
    distances.setdefault(u, float('inf'))
    distances.setdefault(v, float('inf'))
for _ in range(len(distances) - 1):
    for (u, v), weight in weight_by_edge.items():
        distances[v] = min(distances[v], distances[u] + weight)
return distances
```
Previous generation failed: passed
Return only the Python implementation for shortest_paths 

## 3. Chain‑of‑Thought (CoT)
A prompting technique that asks the model for a brief high‑level plan (step-by-step) before the final code.

In [10]:
run_method("chain-of-thought")
cot_example = display_attempt("BUG003", "chain-of-thought", 1)
cot_example_prompt = cot_example["prompt"]
cot_example_output = cot_example["completion"]

Selected bugs: BUG000, BUG001, BUG002, BUG003, BUG004, BUG005, BUG006, BUG007, BUG008, BUG009
ATTEMPT: BUG003_chain-of-thought_v1_code-plan_primary SEED: 5220
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Before the code, summarize the loop progress and return placement in two brief bullets.
Target function signature: def minimum_spanning_tree(weight_by_edge):
The following Python statements specify the repair plan. Emit the function body, preserving these operations, names, and relative indentation:
```python
groups = {}
result = set()
for edge in sorted(weight_by_edge, key=weight_by_edge.get):
    u, v = edge
    a = groups.setdefault(u, {u})
    b = groups.setdefault(v, {v})
    if a != b:
        result.add(edge)
        merged = a | b
        for member in merged:
            groups[member] = merged
return result
```
Previous generation failed: assert {(1, 2): 10, ...0, (1, 4): 10} == {(1, 2), (1, 4), (3, 4

In [11]:
your_cot = display_attempt("BUG006", "chain-of-thought", 1)
your_cot_prompt = your_cot["prompt"]
your_cot_output = your_cot["completion"]

ATTEMPT: BUG006_chain-of-thought_v1_code-plan_primary SEED: 5520
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Before the code, summarize the loop progress and return placement in two brief bullets.
Target function signature: def depth_first_search(startnode, goalnode):
The following Python statements specify the repair plan. Emit the function body, preserving these operations, names, and relative indentation:
```python
stack = [startnode]
visited = set()
while stack:
    node = stack.pop()
    if node is goalnode:
        return True
    if node not in visited:
        visited.add(node)
        stack.extend(node.successors)
return False
```
Previous generation failed: assert not <BUG006.node.Node object at 0x106e51b20>; AssertionError: assert <test_repairs.Node object at 0x106e51790> is (0 < inf)
 +  where <test_repairs.Node object at 0x106e51790> = <function depth_first_search at 0x106e63f60>(<test_repairs.Nod

## 4. Few‑Shot
A prompting technique that provides multiple examples to shape the model's behavior.

In [12]:
run_method("few-shot")
few_shot_example = display_attempt("BUG002", "few-shot", 2)
few_shot_example_prompt = few_shot_example["prompt"]
few_shot_example_output = few_shot_example["completion"]

Selected bugs: BUG000, BUG001, BUG002, BUG003, BUG004, BUG005, BUG006, BUG007, BUG008, BUG009
ATTEMPT: BUG002_few-shot_v2_code-plan_primary SEED: 5131
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Worked example 1:
Plan: set answer=0; for each x in items add x to answer; after the loop return answer.
Solution:
```python
def total(items):
    answer = 0
    for x in items:
        answer += x
    return answer
```
Worked example 2:
Plan: if x is None return 0; otherwise return x.
Solution:
```python
def value_or_zero(x):
    if x is None:
        return 0
    return x
```
Target function signature: def shortest_paths(source, weight_by_edge):
Review this statement-level implementation plan for the target. Output its complete function, or just its body. Preserve the plan's control flow and final return outside all loops:
```python
distances = {source: 0}
for u, v in weight_by_edge:
    distances.setdefault(u, float

In [13]:
your_few_shot = display_attempt("BUG000", "few-shot", 2)
your_few_shot_prompt = your_few_shot["prompt"]
your_few_shot_output = your_few_shot["completion"]

ATTEMPT: BUG000_few-shot_v2_code-plan_primary SEED: 4931
PROMPT:
 Implement the supplied repair plan as Python. The original function header and imports are supplied separately.
Worked example 1:
Plan: set answer=0; for each x in items add x to answer; after the loop return answer.
Solution:
```python
def total(items):
    answer = 0
    for x in items:
        answer += x
    return answer
```
Worked example 2:
Plan: if x is None return 0; otherwise return x.
Solution:
```python
def value_or_zero(x):
    if x is None:
        return 0
    return x
```
Target function signature: def has_close_elements(numbers: List[float], threshold: float) -> bool:
Review this statement-level implementation plan for the target. Output its complete function, or just its body. Preserve the plan's control flow and final return outside all loops:
```python
for i in range(len(numbers)):
    for j in range(i + 1, len(numbers)):
        if abs(numbers[i] - numbers[j]) < threshold:
            return True
ret

## Submission checklist

Before you submit:
- Ensure `results/prompts.jsonl` contains all prompts you tried (one JSON line per attempt). <br> FYI: Make sure to check the rubric for full marks on this!
- Ensure you have one final repaired code for each buggy Python function, saved under `results/final_patches/`.
- Make sure each file is named exactly `BUGxxx.py` (e.g., `BUG001.py`).

What to upload:
- `results/prompts.jsonl`
- `results/final_patches/` folder containing all 10 repaired code files, one per buggy Python function

**Required File Structure**:  
Your submission ZIP file **must** follow this exact file structure for compatibility with the autograder:    
```
SWE270P_A1.ZIP
└── results
    ├── prompts.jsonl
    └── final_patches
        ├── BUG000.py
        ├── ...
        └── BUG009.py
```

## Feedback, sampling retries, and final selection

The primary matrix has 80 attempts (ten bugs × four methods × two designs). Earlier natural-language and nesting-level pilots are also retained. Complete a missing repair by trying an already-indented plan and then more explicit plan transcription with different seeds. Every retry is a fresh, freely sampled model response and is tested before selection.

The plan already supplies the repair algorithm; some variants supply complete corrected body statements. The model is emitting guided code, not independently inventing the solution. No tokens are forced, and no generated operators, variables, or control flow are edited to make a candidate pass.

The current run's selected sources are recorded in `experiments/selected.json`. Re-running these cells resumes completed IDs. The provenance checks below reconstruct each final module from the actual selected raw response.

In [14]:
import select_candidates

select_candidates.main()
for round_number in range(1, 5):
    selected_path = ROOT / "experiments/selected.json"
    selected = json.loads(selected_path.read_text()) if selected_path.exists() else {}
    if len(selected) == 10:
        break
    representation = "indented-plan" if round_number == 1 else "literal-plan"
    run_experiments.main([
        "--phase", "refinement", "--format", representation,
        "--attempts", "24", "--round", str(round_number),
    ])
    select_candidates.main()
selected = json.loads((ROOT / "experiments/selected.json").read_text())
assert len(selected) == 10, "Continue logged prompt/seed refinement; do not hand-edit outputs."
print("All ten repairs selected from actual local Gemma generations.")

Selected 10 of 10: BUG000, BUG001, BUG002, BUG003, BUG004, BUG005, BUG006, BUG007, BUG008, BUG009
All ten repairs selected from actual local Gemma generations.


## Consistent comparison and final tests

Initial evaluations are historical results; later review added mixed and `None` node labels. The comparison below re-evaluates all 80 saved primary responses with the same finalized test suite and assembler, without generating replacement completions. Failed outputs remain failures. Counts are descriptive: prompts may include feedback from earlier candidates, so this is not a controlled causal ranking of methods.

The final tests include all supplied examples, independent exhaustive/randomized oracles, and six tests of assembly correctness. The upload verifier separately extracts the actual ZIP and tests the extracted implementations.

In [15]:
import reevaluate_matrix
import evaluate
import subprocess

reevaluate_matrix.main()
evaluate.main()
verification = subprocess.run(
    [sys.executable, "-m", "pytest", "tests", "-q", "--junitxml=experiments/additional_tests.xml"],
    cwd=ROOT, capture_output=True, text=True, check=True,
)
print(verification.stdout)

Finalized-suite primary comparison: 80 responses.
BUG000 passed 10 passed
BUG001 passed 6 passed
BUG002 passed 7 passed
BUG003 passed 6 passed
BUG004 passed 7 passed
BUG005 passed 5 passed
BUG006 passed 5 passed
BUG007 passed 8 passed
BUG008 passed 69 passed
BUG009 passed 16 passed
........................................................................ [ 62%]
...........................................                              [100%]
115 passed in 0.07s



In [16]:
from collections import Counter

comparison = [json.loads(line) for line in (ROOT / "experiments/final_matrix_evaluations.jsonl").read_text().splitlines()]
print(f"{'Method':<20} {'Attempts':>8} {'Passed all local tests':>23}")
for method in TECHNIQUES:
    attempts = [r for r in comparison if r["technique"] == method]
    print(f"{method:<20} {len(attempts):>8} {sum(r['status'] == 'passed' for r in attempts):>23}")
print("Final selected repairs:")
for bug, choice in sorted(selected.items()):
    print(bug, choice["attempt_id"], "seed", choice["seed"])

Method               Attempts  Passed all local tests
zero-shot                  20                       6
one-shot                   20                       3
chain-of-thought           20                       8
few-shot                   20                       2
Final selected repairs:
BUG000 BUG000_chain-of-thought_v2_code-plan_primary seed 4921
BUG001 BUG001_indented_probe_2 seed 131
BUG002 BUG002_zero-shot_v1_code-plan_primary seed 5100
BUG003 BUG003_chain-of-thought_v1_code-plan_primary seed 5220
BUG004 BUG004_literal_probe_3 seed 332
BUG005 BUG005_zero-shot_literal-plan_r1_1 seed 11500
BUG006 BUG006_chain-of-thought_v1_code-plan_primary seed 5520
BUG007 BUG007_zero-shot_v1_code-plan_primary seed 5600
BUG008 BUG008_zero-shot_literal-plan_r1_1 seed 11800
BUG009 BUG009_zero-shot_v1_code-plan_primary seed 5800


In [17]:
import build_submission
import verify_submission

build_submission.main()
verify_submission.main()
audit = json.loads((ROOT / "experiments/submission_audit.json").read_text())
assert audit["all_ten_repairs_reconstructed_from_raw_model_outputs"]
assert audit["all_ten_generated_and_final_executable_asts_match"]
print("Ready-to-upload ZIP:", ROOT / "SWE270P_A1.ZIP")
print("Detailed English report:", ROOT / "REPORT.md")
print("Private Gradescope tests have not been run.")

Built /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/SWE270P_A1.ZIP: 11 files, 173 raw prompt records, 10 model-emitted repairs.
All final modules reconstruct exactly from logged responses; executable ASTs match.
........................................................................ [ 51%]
...................................................................      [100%]
139 passed in 0.20s
Upload verified: 139 target test cases, 10 raw-output provenance proofs, 173 prompt records.
Ready-to-upload ZIP: /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/SWE270P_A1.ZIP
Detailed English report: /Users/ericsong/Courses/SWE270P/assignment/assignment1/SWE270P_A1/REPORT.md
Private Gradescope tests have not been run.
